# Saliency Maps 

Este notebook:
- Carga un **modelo preentrenado de torchvision** (ResNet50).
- Calcula **Saliency Maps** (|∂score/∂input|) para **todas las imágenes** en una carpeta.
- Guarda resultados en `output_saliency/` como PNG (original, saliency y overlay)

In [1]:
# (Opcional) Si te falta algo en tu entorno, descomenta e instala:
# !pip install -U torch torchvision pillow matplotlib numpy

import os
from pathlib import Path
import numpy as np
from PIL import Image, UnidentifiedImageError
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
if torch.cuda.is_available():
    device = torch.device("cuda")      # o "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")       # Apple Silicon
else:
    device = torch.device("cpu")

print("Device:", device)


Device: mps


In [2]:
# Utilidades: carga segura de imágenes + overlay 

def normalize_0_1(x: np.ndarray, eps=1e-8):
    x = x - x.min()
    x = x / (x.max() + eps)
    return x

def load_image_pil(path: str, size=224):
    """Carga imagen (segura) y devuelve (pil_resized_rgb, tensor_norm_1x3xHxW)."""
    try:
        # verify detecta muchos archivos corruptos
        with Image.open(path) as im:
            im.verify()
        pil = Image.open(path).convert("RGB")
        pil_resized = pil.resize((size, size))

        img = np.array(pil_resized).astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_norm = (img - mean) / std

        tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0)  # 1x3xHxW
        return pil_resized, tensor.to(device)

    except (UnidentifiedImageError, OSError) as e:
        print(f"⚠️ Saltando archivo no válido: {path} | {e}")
        return None, None

def overlay_map_on_image(pil_img: Image.Image, m_0_1: np.ndarray, alpha=0.45, cmap_name="jet"):
    """Devuelve (map_rgb_uint8, overlay_rgb_uint8)."""
    m_0_1 = np.clip(m_0_1, 0, 1)
    cmap = plt.get_cmap(cmap_name)
    heat = cmap(m_0_1)[:, :, :3]  # HxWx3 float [0,1]

    img = np.array(pil_img).astype(np.float32) / 255.0
    overlay = (1 - alpha) * img + alpha * heat
    overlay = np.clip(overlay, 0, 1)

    heat_u8 = (heat * 255).astype(np.uint8)
    overlay_u8 = (overlay * 255).astype(np.uint8)
    return heat_u8, overlay_u8


In [3]:
# Cargar modelo torchvision (ResNet50 preentrenado)

try:
    from torchvision.models import resnet50, ResNet50_Weights
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)
    imagenet_labels = weights.meta.get("categories", None)
except Exception:
    from torchvision.models import resnet50
    model = resnet50(pretrained=True)
    imagenet_labels = None

model = model.to(device).eval()

print("Modelo listo:", model.__class__.__name__)


Modelo listo: ResNet


In [4]:
# Saliency Map (gradiente del score respecto al input)

def saliency_map(model, x, class_idx=None):
    """Devuelve (saliency_0_1, class_idx, score_logit)."""
    # x: 1x3xHxW (normalizado). Queremos gradiente sobre x.
    x = x.clone().detach().requires_grad_(True)

    model.zero_grad(set_to_none=True)
    logits = model(x)

    if class_idx is None:
        class_idx = int(torch.argmax(logits, dim=1).item())

    score = logits[0, class_idx]
    score.backward()

    # Gradiente del input: 1x3xHxW
    grad = x.grad.detach()

    # Saliency clásico: max(|grad|) sobre canales -> HxW
    sal = grad.abs().max(dim=1)[0][0].cpu().numpy()
    sal = normalize_0_1(sal)

    return sal, class_idx, float(score.detach().cpu().item())


In [5]:
# CONFIG: carpeta de entrada y salida

INPUT_DIR = Path("../imagenes/original")    
OUTPUT_DIR = Path("../imagenes/output_saliency")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Opcional: limitar cuántas procesa (None = todas)
MAX_IMAGES = None

print("INPUT_DIR:", INPUT_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


INPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/original
OUTPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/output_saliency


In [6]:
# Ejecutar Saliency Maps sobre todas las imágenes de la carpeta

paths = [p for p in sorted(INPUT_DIR.rglob("*")) if p.suffix.lower() in EXTS]
if MAX_IMAGES is not None:
    paths = paths[:MAX_IMAGES]

print("Imágenes encontradas:", len(paths))
if len(paths) == 0:
    raise FileNotFoundError(f"No encontré imágenes en {INPUT_DIR}. Revisa INPUT_DIR.")

processed = 0

for i, p in enumerate(paths, 1):
    pil_img, x = load_image_pil(str(p), size=224)
    if pil_img is None:
        continue

    sal_0_1, cls_idx, score = saliency_map(model, x, class_idx=None)

    if imagenet_labels is not None and 0 <= cls_idx < len(imagenet_labels):
        cls_name = imagenet_labels[cls_idx]
    else:
        cls_name = str(cls_idx)

    heat_u8, overlay_u8 = overlay_map_on_image(pil_img, sal_0_1, alpha=0.45, cmap_name="jet")

    out_name = OUTPUT_DIR / f"{p.stem}_saliency.png"
    fig = plt.figure(figsize=(9, 3))

    ax1 = plt.subplot(1, 3, 1)
    ax1.imshow(pil_img)
    ax1.set_title("Original")
    ax1.axis("off")

    ax2 = plt.subplot(1, 3, 2)
    ax2.imshow(heat_u8)
    ax2.set_title("Saliency" )
    ax2.axis("off")

    ax3 = plt.subplot(1, 3, 3)
    ax3.imshow(overlay_u8)
    ax3.set_title(f"Overlay\n{cls_name} (logit={score:.2f})")
    ax3.axis("off")

    plt.tight_layout()
    fig.savefig(out_name, bbox_inches="tight")
    plt.close(fig)

    processed += 1
    if processed % 10 == 0 or i == len(paths):
        print(f"Procesadas: {processed} | Último guardado: {out_name}")

print("Revisa la carpeta output_saliency/")


Imágenes encontradas: 11
Procesadas: 10 | Último guardado: ../imagenes/output_saliency/image10_saliency.png
Procesadas: 11 | Último guardado: ../imagenes/output_saliency/image11_saliency.png
Revisa la carpeta output_saliency/
